In [ ]:
from datetime import date
from functools import partial
import pandas as pd
import sklearn
import sklearn.preprocessing
import matplotlib.pyplot as plt
from pl_pairs_trading import *

# Gen data

In [ ]:
def gen_lvls(mu, sig, n_pts=1000):
    return pd.Series(1. + (
        mu + sig * np.random.randn(n_pts)).cumsum())

In [ ]:
n_assets = 50
mus = np.linspace(-0.2, 0.2, n_assets) / 365
sig = 0.1 / 365 ** 0.5
levels = [gen_lvls(mu, sig) for mu in mus]
# levels[10].plot(grid=True)

dates = (
    pd.Series(pd.date_range(start='1/1/2000', periods=len(levels[0])))
    .rename('date')
)
levels = pd.concat([dates] + levels, axis=1).set_index('date')
levels.plot(grid=True, legend=False)

In [ ]:
levels = (
    pl.DataFrame(levels.reset_index())
    .with_columns(pl.col('date').cast(pl.Date))
)

In [ ]:
cut = date(2001, 1, 1)
formation = levels.filter(pl.col('date') <= cut)
trading = levels.filter(pl.col('date') > cut)

In [ ]:
rule = partial(trading_rule, threshold=1)
preprocessor_name = 'StandardScaler'
top = 20
kwargs = {}

# Test

In [ ]:
%%time
positions = fit_trade(formation, preprocessor_name, top, trading, rule, **kwargs)

In [ ]:
# %%timeit
# positions = fit_trade(formation, preprocessor_name, top, trading, rule, **kwargs)
# 995 ms ± 90.2 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)

# Rolling

In [ ]:
gcd = '5d'

In [ ]:
dates = pd.Series(index=dates, data=0)
cuts = dates.resample(gcd).first().index
splits = [
    ((start, start_2), (start_2, start_3))
    for start, start_2, start_3
    in zip(cuts, cuts[2:], cuts[3:])
]

train, test = splits[0]
formation = levels.filter((train[0] <= pl.col('date')) & (pl.col('date') < train[1]))
trading = levels.filter((test[0] <= pl.col('date')) & (pl.col('date') < test[1]))
fit_trade(formation, preprocessor_name, top, trading, rule, **kwargs).tail()

In [ ]:
period = '120d'
every = '60d'

dates = (
    levels
    .with_columns(pl.col('date').alias('date_'))
    .groupby_dynamic('date', period=period, every=every)
    .agg(pl.col('date_').last().alias('end_train'))
    .select([
        pl.col('date').alias('start_train'),
        pl.col('end_train'),
        pl.col('date').shift(-2).alias('start_test'),
        pl.col('end_train').shift(-1).alias('end_test')
    ])
    .head(-2)
)
dates.tail()

In [ ]:
%%time
positions = [
    fit_trade(
        levels.filter((dates_['start_train'] <= pl.col('date')) & (pl.col('date') <= dates_['end_train'])),
        preprocessor_name,
        top,
        levels.filter((dates_['start_test'] <= pl.col('date')) & (pl.col('date') <= dates_['end_test'])),
        rule,
        **kwargs
    )
    for dates_ in dates.iter_rows(named=True)
]

In [ ]:
positions = to_unit_leverage(simple_agg(positions, ['date', 'asset']), 'position')

plot_ts(positions.groupby('date').agg(pl.col('position').abs().sum()))

In [ ]:
descs = positions.groupby('date').agg(window_describe('position'))
plot_ts(descs.drop('count'))